# R3_01 — Label silver baru dan train_pairs (score_sentence, θ = 15,0)

* Memakai **sel asli** `notebooks/02_Silverrr_Labeling_dan_SentencePair.ipynb` (salinan verbatim di `legacy_code/`):
  leksikon, `score_sentence`, `score_windowed`, `detect_aspect`, **loop pelabelan**, `ASPECT_DESCRIPTIONS`, `build_pairs`.
  Varian `score_sentence` dibuat dengan mengganti fungsi skor yang dipanggil loop, tanpa mengubah satu baris pun di loop
  (sama dengan `revisi_round3/calibrate_method_theta_validation.ipynb`).
* Verifikasi: (1) loop yang sama dengan konfigurasi lama mereproduksi `data/silver/silver_labeled.csv` 100%; (2) distribusi baru
  **persis** 93.203 / 4.758 / 4.812 / 48.215 dan **29.284** label berubah — bila tidak cocok, notebook berhenti.
* Output (baru, tidak menimpa file lama): `data_round3/silver_labeled_s15.csv`, `data_round3/train_pairs.csv`, `data_round3/R3_01_manifest.json`.
* Tidak memuat data uji. Waktu: ±1–2 menit.

In [ ]:
import os, sys, json
sys.path.insert(0, os.environ.get('R3_CODE_DIR', os.getcwd()))
import numpy as np, pandas as pd
import config as C
import r3_lib as L
L.forbid_test_access()
C.DATA_R3.mkdir(parents=True, exist_ok=True)
print('Metode:', C.SILVER_METHOD, '| θ =', C.SILVER_THETA, '| output:', C.DATA_R3)

In [ ]:
df = pd.read_csv(C.TWEETS_PREPROCESSED); df['tweet_id'] = df['tweet_id'].astype(str)
gold_ids = set(pd.read_csv(C.GOLD_TWEET_IDS, dtype=str).tweet_id)
silver_old = pd.read_csv(C.SILVER_OLD); silver_old['tweet_id'] = silver_old['tweet_id'].astype(str)
train_old = pd.read_csv(C.TRAIN_PAIRS_OLD, dtype={'tweet_id': str})
assert len(gold_ids) == 200 and len(silver_old) == len(train_old) == C.EXPECTED_N_PAIRS
df_for_silver = df[~df['tweet_id'].isin(gold_ids)].copy()
assert df_for_silver.tweet_id.nunique() == 37747
print(f'{len(df):,} tweet; {len(df_for_silver):,} tweet non-gold untuk silver')

In [ ]:
ns = L.base_namespace(LEXICON_DIR=os.path.join(C.RESOURCES, 'inset_lexicon'),
                      KEYWORDS_PATH=os.path.join(C.RESOURCES, 'aspect_keywords.json'))
L.exec_legacy(['nb02_load_lexicon', 'nb02_scorer', 'nb02_score_windowed', 'nb02_detect_aspect'], ns)
LABELING_SRC = L.legacy_source('nb02_labeling_loop')
print('Sel loop pelabelan asli (sha256', L.sha256_text(LABELING_SRC)[:16], '…)')

def run_labeling_loop(tweets, method, theta):
    """Eksekusi sel loop pelabelan ASLI; untuk score_sentence, fungsi skor yang dipanggil loop diganti."""
    loop_ns = dict(ns); loop_ns['df_for_silver'] = tweets.copy(); loop_ns['BEST_THETA'] = theta
    if method == 'score_sentence':
        loop_ns['score_windowed'] = lambda text, aspect, window=5: ns['score_sentence'](text)
    else:
        assert method == 'score_windowed'
    exec(compile(LABELING_SRC, '<legacy:nb02_labeling_loop>', 'exec'), loop_ns)
    return loop_ns['df_silver']

## Verifikasi 1 — kode yang sama mereproduksi label lama

In [ ]:
old_regen = run_labeling_loop(df_for_silver, 'score_windowed', 3.0)
m = old_regen.merge(silver_old, on=['tweet_id', 'aspek'], suffixes=('_ulang', '_file'))
assert len(m) == C.EXPECTED_N_PAIRS
assert (m.label_silver_ulang == m.label_silver_file).all() and np.allclose(m.lexicon_score_ulang, m.lexicon_score_file)
print('✓ Loop asli (score_windowed, θ=3,0) mereproduksi data/silver/silver_labeled.csv 100%')

## Label baru (D1–D2) dan verifikasi 2

In [ ]:
silver_new = run_labeling_loop(df_for_silver, C.SILVER_METHOD, C.SILVER_THETA)
dist = silver_new.label_silver.value_counts().to_dict()
cmp = silver_new.merge(silver_old[['tweet_id', 'aspek', 'label_silver']], on=['tweet_id', 'aspek'], suffixes=('', '_lama'))
n_changed = int((cmp.label_silver != cmp.label_silver_lama).sum())
print('Distribusi baru:', dist, '| label berubah:', n_changed)
if dist != C.EXPECTED_SILVER_DIST or n_changed != C.EXPECTED_N_CHANGED or len(silver_new) != C.EXPECTED_N_PAIRS:
    raise RuntimeError(f'TIDAK COCOK dengan rencana: distribusi {dist} (harus {C.EXPECTED_SILVER_DIST}), '
                       f'berubah {n_changed} (harus {C.EXPECTED_N_CHANGED})')
print('✓ Distribusi dan jumlah perubahan cocok dengan rencana_latih_ulang.md')
pd.crosstab(cmp.label_silver_lama, cmp.label_silver, margins=True)

## train_pairs baru (build_pairs asli, sentence_a = text_v1)

In [ ]:
pairs_ns = L.base_namespace(df=df)
L.exec_legacy(['nb02_aspect_desc', 'nb02_build_pairs'], pairs_ns)
train_new = pairs_ns['build_pairs'](silver_new, label_col='label_silver', text_version=C.TEXT_VERSION)
assert len(train_new) == C.EXPECTED_N_PAIRS
train_new['tweet_id'] = train_new['tweet_id'].astype(str)
# Identik dengan train_pairs lama kecuali label
assert (train_new.pair_id.values == train_old.pair_id.values).all()
assert (train_new.tweet_id.values == train_old.tweet_id.astype(str).values).all()
assert (train_new.sentence_a.values == train_old.sentence_a.astype(str).values).all()
assert (train_new.sentence_b.values == train_old.sentence_b.values).all()
n_lab_changed = int((train_new.label.values != train_old.label.values).sum())
assert n_lab_changed == C.EXPECTED_N_CHANGED
print(f'✓ train_pairs baru: urutan, pair_id, teks identik dengan yang lama; {n_lab_changed:,} label berubah')
print(train_new.label.map(C.LABEL_MAP_INV).value_counts().to_dict())

## Simpan + sha256

In [ ]:
def save_once(frame, path):
    """Tulis file; bila sudah ada, isinya harus identik (tidak pernah menimpa dengan isi berbeda)."""
    path = C.assert_writable(path)
    tmp = path.with_suffix('.tmp.csv')
    frame.to_csv(tmp, index=False, encoding='utf-8-sig')
    if path.exists():
        same = L.sha256_file(tmp) == L.sha256_file(path)
        tmp.unlink()
        if not same:
            raise RuntimeError(f'{path} sudah ada dengan isi berbeda — tidak ditimpa')
        print(f'  (sudah ada, identik) {path.name}')
    else:
        os.replace(tmp, path)
        print(f'  [saved] {path.name}')

save_once(silver_new[['tweet_id', 'text', 'aspek', 'label_silver', 'lexicon_score']], C.SILVER_R3)
save_once(train_new, C.TRAIN_PAIRS_R3)
manifest = {
    'created_at': L.now_utc(), 'method': C.SILVER_METHOD, 'theta': C.SILVER_THETA, 'text_version': C.TEXT_VERSION,
    'distribution': dist, 'n_changed_vs_old': n_changed,
    'labeling_cell_sha256': L.sha256_text(LABELING_SRC),
    'outputs': {p.name: L.sha256_file(p) for p in [C.SILVER_R3, C.TRAIN_PAIRS_R3]},
    'inputs': {p.name: L.sha256_file(p) for p in [C.TWEETS_PREPROCESSED, C.GOLD_TWEET_IDS, C.SILVER_OLD, C.TRAIN_PAIRS_OLD,
                                                  C.RESOURCES / 'aspect_keywords.json', C.RESOURCES / 'inset_lexicon/positive.tsv',
                                                  C.RESOURCES / 'inset_lexicon/negative.tsv']},
    'environment': L.environment_info(),
}
L.write_json(C.DATA_R3 / 'R3_01_manifest.json', manifest)
json.dumps(manifest['outputs'], indent=1)